In [1]:
import pandas as pd
import re
from pathlib import Path
from collections import Counter
import subprocess

# PLINK v1.90b7 64-bit (16 Jan 2023)
# GERMLINE
# bfctools

In [2]:
# generate the ground truth data
DATA_DIR = Path(".")
GT_TXT = DATA_DIR / "20140625_related_individuals.txt"

def canonical_pair(a: str, b: str):
    return (a, b) if a <= b else (b, a)

def parse_reason(sample_id: str, reason: str):
    reason = str(reason).strip()
    out = []

    # Sibling
    m = re.match(r"^Sibling:(\S+)$", reason)
    if m:
        partner = m.group(1)
        a, b = canonical_pair(sample_id, partner)
        out.append((a, b, "Sibling"))
        return out

    # Parent
    m = re.match(r"^Parent:(\S+)$", reason)
    if m:
        partner = m.group(1)
        a, b = canonical_pair(sample_id, partner)
        out.append((a, b, "Parent"))
        return out

    # Second Order
    m = re.match(r"^Second\s+Order:(\S+)$", reason)
    if m:
        partner = m.group(1)
        a, b = canonical_pair(sample_id, partner)
        out.append((a, b, "SecondOrder"))
        return out

    # Trio
    m = re.match(r"^trio:(\S+),(\S+)$", reason)
    if m:
        p1, p2 = m.group(1), m.group(2)
        a, b = canonical_pair(sample_id, p1)
        out.append((a, b, "Trio"))
        a, b = canonical_pair(sample_id, p2)
        out.append((a, b, "Trio"))
        return out

    return out


# 1) Read file
df = pd.read_csv(GT_TXT, sep=r"\t+", engine="python")

df.columns = df.columns.str.strip()

print("Columns:", df.columns.tolist())

# 2) Build pairs
pairs = []
unknown_reasons = Counter()

for _, row in df.iterrows():
    sid = str(row["Sample"]).strip()
    reason = row["Reason for exclusion"]
    extracted = parse_reason(sid, reason)
    if extracted:
        pairs.extend(extracted)
    else:
        unknown_reasons[str(reason).strip()] += 1

pairs_df = (
    pd.DataFrame(pairs, columns=["id1", "id2", "relation"])
    .drop_duplicates()
    .sort_values(["id1", "id2", "relation"])
)

# 3) Write files
pairs_df.to_csv("gt_pairs.tsv", sep="\t", index=False, header=False)

keep_ids = sorted(set(pairs_df["id1"]).union(set(pairs_df["id2"])))
with open("keep.samples", "w") as f:
    for x in keep_ids:
        f.write(x + "\n")

print("gt_pairs.tsv written. n_pairs =", len(pairs_df))
print("keep.samples written. n_samples =", len(keep_ids))
print("\nRelation counts:")
print(pairs_df["relation"].value_counts())

pairs_df.head(10)

Columns: ['Sample', 'Population', 'Gender', 'Reason for exclusion']
gt_pairs.tsv written. n_pairs = 35
keep.samples written. n_samples = 65

Relation counts:
relation
Sibling        11
SecondOrder     9
Trio            8
Parent          7
Name: count, dtype: int64


,id1,id2,relation
0,HG00119,HG00124,SecondOrder
1,HG00501,HG00524,Sibling
2,HG00581,HG00635,Sibling
3,HG00658,HG00702,Sibling
4,HG00731,HG00733,Trio
5,HG00732,HG00733,Trio
6,HG01936,HG01983,SecondOrder
7,HG02024,HG02025,Trio
8,HG02024,HG02026,Trio
9,HG02046,HG02067,Sibling


In [5]:
# find missing individuals
CHRS = [1,2,3,4,5]

VCF_FULL_TMPL = "ALL.chr{c}.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz"
VCF_REL_TMPL  = "ALL.chr{c}.phase3_shapeit2_mvncall_integrated_v5_related_samples.20130502.genotypes.vcf.gz"

KEEP = Path("keep.samples")

# assert VCF_FULL.exists(), VCF_FULL
# assert VCF_REL.exists(), VCF_REL
# assert KEEP.exists(), KEEP


keep_ids = [x.strip() for x in KEEP.read_text().splitlines() if x.strip()]

import subprocess
from pathlib import Path

def vcf_samples(vcf_path: Path):
    out = subprocess.check_output(["bcftools", "query", "-l", str(vcf_path)], text=True)
    return [x.strip() for x in out.splitlines() if x.strip()]

VCF_FULL = Path(VCF_FULL_TMPL.format(c=1))
VCF_REL  = Path(VCF_REL_TMPL.format(c=1))

full_ids = set(vcf_samples(VCF_FULL))
rel_ids  = set(vcf_samples(VCF_REL))
keep_set = set(keep_ids)

in_full = sorted(keep_set & full_ids)
in_rel  = sorted(keep_set & rel_ids)
missing = sorted(keep_set - (full_ids | rel_ids))

print("Using chr1 to check missing:")
print("In FULL:", len(in_full))
print("In REL :", len(in_rel))
print("Missing:", len(missing))
if missing:
    print("Missing sample IDs:", missing[:20])

Using chr1 to check missing:
In FULL: 33
In REL : 31
Missing: 1
Missing sample IDs: ['HG00658']


In [6]:
# remove the missing individual from the groud truth tsv
missing = {"HG00658"}  

# 1) keep.present
keep = [x.strip() for x in open("keep.samples") if x.strip()]
keep_present = [x for x in keep if x not in missing]

with open("keep.present", "w") as f:
    for x in keep_present:
        f.write(x + "\n")

print("keep.samples:", len(keep))
print("keep.present:", len(keep_present))
print("removed:", sorted(set(keep) - set(keep_present)))

# 2) gt_pairs.present.tsv
gt = pd.read_csv("gt_pairs.tsv", sep="\t", header=None, names=["id1","id2","relation"])
gt_present = gt[~gt["id1"].isin(missing) & ~gt["id2"].isin(missing)].copy()
gt_present.to_csv("gt_pairs.present.tsv", sep="\t", index=False, header=False)

print("gt_pairs.tsv:", len(gt))
print("gt_pairs.present.tsv:", len(gt_present))
gt_present["relation"].value_counts()

keep.samples: 65
keep.present: 64
removed: ['HG00658']
gt_pairs.tsv: 35
gt_pairs.present.tsv: 34


relation
Sibling        10
SecondOrder     9
Trio            8
Parent          7
Name: count, dtype: int64

In [7]:
VCF_FULL = VCF_FULL_TMPL.format(c=1)
VCF_REL  = VCF_REL_TMPL.format(c=1)

def vcf_samples(vcf_path: str):
    out = subprocess.check_output(["bcftools", "query", "-l", vcf_path], text=True)
    return set([x.strip() for x in out.splitlines() if x.strip()])

keep_present_set = set([x.strip() for x in open("keep.present") if x.strip()])
full_ids = vcf_samples(VCF_FULL)
rel_ids  = vcf_samples(VCF_REL)

take_from_full = sorted(keep_present_set & full_ids)
take_from_rel  = sorted(keep_present_set & rel_ids)

Path("keep.present.from_full").write_text("\n".join(take_from_full) + ("\n" if take_from_full else ""))
Path("keep.present.from_rel").write_text("\n".join(take_from_rel) + ("\n" if take_from_rel else ""))

print("Take from FULL:", len(take_from_full))
print("Take from REL :", len(take_from_rel))
print("Union:", len(set(take_from_full) | set(take_from_rel)))
print("Expected keep.present:", len(keep_present_set))

Take from FULL: 33
Take from REL : 31
Union: 64
Expected keep.present: 64


In [8]:
# Use command lines in terminal to merge chr1-5

In [9]:
# Use command lines in terminal to run plinks
!source /home/jovyan/.bashrc

In [10]:
# Store the output from plink
PLINK_GENOME = "plink_genome.present.chr1_5.qcpruned.genome"

gen = pd.read_csv(PLINK_GENOME, sep=r"\s+", engine="python")
gen.to_csv("plink_genome.present.chr1_5.qcpruned.tsv", sep="\t", index=False)

In [11]:
# Plink output
# use plink (z1+z2) to find relationship (First/second degree)

gen["Z1Z2"] = gen["Z1"] + gen["Z2"]

def canonical_pair(a, b):
    return (a, b) if a <= b else (b, a)

# PC_thres = 0.95
# sib_thres = 0.70
# second_thres = 0.45

First_thres = 0.7
Second_thres = 0.4

rows = []
for _, r in gen.iterrows():
    score = float(r["Z1Z2"])
    if score >= First_thres:
        rel = "First"
    elif score >= Second_thres:
        rel = "Second"
    # elif score >= TH_SECOND:
    #     rel = "SecondOrder"
    else:
        continue
    
    a, b = canonical_pair(str(r["IID1"]), str(r["IID2"]))
    rows.append((a, b, rel))

plink_z1z2 = (
    pd.DataFrame(rows, columns=["id1","id2","relation"])
    .drop_duplicates()
    .sort_values(["id1","id2"])
)

plink_z1z2.to_csv("plink_chr1_5_z1z2.present.tsv", sep="\t", index=False, header=False)
print("wrote plink_chr1_5_z1z2.present.tsv")
print("n_pred =", len(plink_z1z2))
plink_z1z2, plink_z1z2.relation.value_counts()

wrote plink_chr1_5_z1z2.present.tsv
n_pred = 34


(        id1      id2 relation
 0   HG00119  HG00124   Second
 1   HG00501  HG00524    First
 2   HG00581  HG00635    First
 3   HG00731  HG00733    First
 4   HG00732  HG00733    First
 5   HG01936  HG01983   Second
 6   HG02024  HG02025    First
 7   HG02024  HG02026    First
 8   HG02046  HG02067    First
 9   HG02250  HG02377    First
 10  HG02353  HG02363   Second
 11  HG02371  HG02372    First
 12  HG02373  HG02381    First
 13  HG02375  HG02388   Second
 14  HG02386  HG02387   Second
 15  HG03673  HG03948    First
 16  HG03713  HG03715    First
 17  NA19027  NA19311   Second
 18  NA19238  NA19240    First
 19  NA19239  NA19240    First
 20  NA19313  NA19331    First
 21  NA19660  NA19664   Second
 22  NA19660  NA19685    First
 23  NA19661  NA19685    First
 24  NA19675  NA19678    First
 25  NA19675  NA19679    First
 26  NA19713  NA19985   Second
 27  NA20289  NA20341    First
 28  NA20321  NA20322    First
 29  NA20334  NA20336    First
 30  NA20526  NA20792    First
 31  NA2

In [12]:
# Use command lines in terminal to run GERMLINE

In [14]:
# Store output of GERMLINE
MAP = "gt_only.present.chr1_5.qcpruned.nomissSNP_ped.map"
m = pd.read_csv(MAP, sep=r"\s+", header=None, engine="python")
# PLINK .map columns: 0=chr, 1=snp_id, 2=genetic_dist, 3=bp
m[0] = m[0].astype(str)
m[3] = m[3].astype(int)

# Sum per-chromosome physical span (in Mb)
span_bp = (m.groupby(0)[3].agg(["min","max"]))
span_bp["span"] = span_bp["max"] - span_bp["min"] + 1
L_mb = span_bp["span"].sum() / 1e6

print("L_mb (chr1-5 total span) =", L_mb)
span_bp.head()

MATCH = "germline.chr1_5.qcpruned.perchr_merged.match"

df = pd.read_csv(MATCH, sep=r"\s+", header=None, engine="python")
df.to_csv("germline_chr1_5_match.tsv", sep="\t", index=False, header=False)

print("Units:", df[11].astype(str).unique())


L_mb (chr1-5 total span) = 1062.043339
Units: <StringArray>
['MB']
Length: 1, dtype: str


In [15]:
id1 = df[1].astype(str)     # IID1
id2 = df[3].astype(str)     # IID2
seg_len_mb = df[10].astype(float)

def canon(a, b):
    return (a, b) if a <= b else (b, a)

pairs = [canon(a, b) for a, b in zip(id1, id2)]

seg = pd.DataFrame(pairs, columns=["id1","id2"])
seg["seg_len_mb"] = seg_len_mb

agg = (seg.groupby(["id1","id2"])
          .agg(total_ibd_mb=("seg_len_mb","sum"),
               n_seg=("seg_len_mb","size"),
               max_seg_mb=("seg_len_mb","max"))
          .reset_index())

# Normalize by total Mb span across chr1-5
agg["z1z2_hat"] = agg["total_ibd_mb"] / L_mb

agg = agg.sort_values(["z1z2_hat","total_ibd_mb","n_seg"], ascending=False)

agg.to_csv("germline_pairs_norm.present.chr1_5.tsv", sep="\t", index=False)
print("wrote germline_pairs_norm.present.chr1_5.tsv, n_pairs =", len(agg))

agg.head(15)

wrote germline_pairs_norm.present.chr1_5.tsv, n_pairs = 205


,id1,id2,total_ibd_mb,n_seg,max_seg_mb,z1z2_hat
195,NA20289,NA20341,94.368,22,7.251,0.088855
7,HG00501,HG00524,64.877,16,8.881,0.061087
129,HG02373,HG02381,58.650,13,7.618,0.055224
199,NA20334,NA20336,36.782,9,6.753,0.034633
196,NA20321,NA20322,36.535,10,5.007,0.034401
103,HG02046,HG02067,33.387,8,5.955,0.031437
194,NA19713,NA19985,32.077,9,4.073,0.030203
23,HG00581,HG00635,27.714,7,5.514,0.026095
203,NA20886,NA20898,24.947,2,21.159,0.023490
92,HG02025,NA20886,21.516,1,21.516,0.020259
